# WikiConv Create Conversations

This notebook provides different forms of conversations from  the Wikiconv data set. In particular, it showcases the final version of selected conversations and how that conversation developed over time. It also provides a framework to print out rnadomly selected final conversations and the corresponding wikipedia page.

In [39]:
#import relevant modules
from datetime import datetime, timedelta
from convokit import Corpus, download
import re
import random

In [40]:
# Load the 2003 wikiconv corpus (feel free to change this to a year of your preference)
wikiconv_corpus = Corpus(filename=download('wikiconv-german-2004'))

Some basic facts about this subset of the corpus: 91,787 conversations and 140,265 utterances

In [41]:
len(list(wikiconv_corpus.iter_conversations()))

108521

In [42]:
len(list(wikiconv_corpus.iter_utterances()))

216963

From the corpus, we will randomly select conversations to print out based on a few metrics:
1. number_of_conversations - how many conversations we want to print out
2. conversation_min_length -  the minimum number of utterances we want in the conversation

In [43]:
#Randomly chooses the set number of conversations to print from the entire conversaton set
def print_random_conversations(conversation_list, number_of_conversations, conversation_min_length,  conversation_corpus): 
    randomly_generated_conversation_list = []
    while (len(randomly_generated_conversation_list) != number_of_conversations):
        new_conversation = random.randint(0, (len(conversation_list)-1))
        new_conversation_id = conversation_list[new_conversation]
        conversation_ids_list = new_conversation_id.get_utterance_ids()
        if (new_conversation not in randomly_generated_conversation_list 
            and (len(conversation_ids_list) >= conversation_min_length)):
            randomly_generated_conversation_list.append(new_conversation_id)
        
    return randomly_generated_conversation_list


Here we'll get a set of random conversatinos from the corpus based on our specifications (print out 3, conversations, with a minimum length of 2) and the output will be a set of serialized conversations.

In [44]:
conversation_list = list(wikiconv_corpus.iter_conversations())
number_of_conversations_to_print = 3
conversation_min_length = 2

random_conversations = print_random_conversations(conversation_list, number_of_conversations_to_print,
                                                     conversation_min_length, wikiconv_corpus)
print (random_conversations)

[Conversation({'obj_type': 'conversation', 'vectors': [], 'tree': None, 'owner': <convokit.model.corpus.Corpus object at 0x17674c8e0>, 'id': '1689329.121.121', 'meta': ConvoKitMeta({'page_id': '252882', 'page_title': 'Frank Jacobsen', 'page_type': 'user_talk'})}), Conversation({'obj_type': 'conversation', 'vectors': [], 'tree': None, 'owner': <convokit.model.corpus.Corpus object at 0x17674c8e0>, 'id': '3542060.186690.186690', 'meta': ConvoKitMeta({'page_id': '56773', 'page_title': 'Studentenverbindung', 'page_type': 'talk'})}), Conversation({'obj_type': 'conversation', 'vectors': [], 'tree': None, 'owner': <convokit.model.corpus.Corpus object at 0x17674c8e0>, 'id': '2330378.9977.9977', 'meta': ConvoKitMeta({'page_id': '96615', 'page_title': 'Stw', 'page_type': 'user_talk'})})]


Next, stored in the conversation meta data is the wikipedia information from the page that this conversation is from.
We will find that information and print out the link to the associated wikipedia page for each conversation.


In [45]:
def wikipedia_link_info(conversation):
    page_title = conversation.meta['page_title']
    page_title = re.sub('\s+', '_', page_title)
    page_type = conversation.meta['page_type']
    link_value = "https://en.wikipedia.org/w/index.php?title="+page_type+":"+page_title
    
    return link_value

for conversation in random_conversations:
    print(wikipedia_link_info(conversation))
    conversation_ids_list = conversation.get_utterance_ids()

https://en.wikipedia.org/w/index.php?title=user_talk:Frank_Jacobsen
https://en.wikipedia.org/w/index.php?title=talk:Studentenverbindung
https://en.wikipedia.org/w/index.php?title=user_talk:Stw


Now that we have the conversation and the actual wikipedia page where they exist, we will want to print out the conversation's final form from the utterance data. But to do this, first we will need to compute the correct order of the comments. 

The corpus functionality does not guarantee the comments are in the right order, so we will compute this flow now.


In [46]:
#For any comments that do not have matching reply to ids, sort these comments in order of recency 
def sort_by_timestamp(conversation_ids_list, conversation_corpus):
    list_of_utterances = []
    for id_val in conversation_ids_list:
        utterance_value = conversation_corpus.get_utterance(id_val)
        timestamp_val = utterance_value.timestamp
        tuple_val = (id_val, timestamp_val)
        list_of_utterances.append(tuple_val)

    sorted_utterance_list = sorted(list_of_utterances, key = lambda x:x[1])
    sorted_utterance_list.reverse()
    id_list = [i[0] for i in sorted_utterance_list]
    return (id_list)

In [47]:
#Find cases in which an utterance's reply to is to a comment in the chain that has been modified, deleted or restored
def check_lists_for_match(x, conversation_ids_list, utterance, next_utterance_value, conversation_corpus):
    modification_list = utterance.meta['modification']
    deletion_list = utterance.meta['deletion']
    restoration_list = utterance.meta['restoration']
    if (len(modification_list)>0):
        for utterance_val in modification_list:
            if (utterance_val['id'] == next_utterance_value.reply_to):
                conversation_ids_list.insert(x+1, next_utterance_value.id)
    if (len(deletion_list)>0):
        for utterance_val in deletion_list:
            if (utterance_val['id'] == next_utterance_value.reply_to):
                conversation_ids_list.insert(x+1, next_utterance_value.id)
    if (len(restoration_list)>0):
        for utterance_val in restoration_list:
            if (utterance_val['id'] == next_utterance_value.reply_to):
                conversation_ids_list.insert(x+1, next_utterance_value.id)

In [48]:
# Build the conversation flow correctly and add utterances if the reply-to id matches the current utterance in the list
def add_utterance(conversation_ids_list, next_utterance_value, conversation_corpus):
    if next_utterance_value.id in conversation_ids_list:
        return conversation_ids_list
    elif (next_utterance_value.reply_to is None):
        conversation_ids_list.append(next_utterance_value.id)
    else:
        for x in range(0,len(conversation_ids_list)):
            utterance_id = conversation_ids_list[x]
            if (utterance_id == next_utterance_value.reply_to):
                conversation_ids_list.insert(x+1, next_utterance_value.id)
            else:
                check_lists_for_match(x, conversation_ids_list, conversation_corpus.get_utterance(utterance_id), next_utterance_value, conversation_corpus)

    return conversation_ids_list

In [49]:
#The order of the returned conversation ids is not guaranteed; compute the correct ordering 
def find_correct_order(conversation_ids_list, conversation_corpus):
    correct_list_order = []
    #if the conversation has only one comment, return the conversation list
    if (len(conversation_ids_list) == 1 ):
        return conversation_ids_list

    #When the conversation has more than one comment, find the correct order of the comments
    if (len(conversation_ids_list) >1):
        #Implement a fail safe to efficiently sort 
        number_of_iterations = 0
        while (number_of_iterations <20 and len(correct_list_order) != len(conversation_ids_list)):
            for utterance_id in conversation_ids_list:
                correct_list_order = add_utterance(correct_list_order, conversation_corpus.get_utterance(utterance_id), conversation_corpus)
            number_of_iterations+=1

        #In some of the conversations, new utterances will be added that don't reply directly to the current conversation
        #Instead, these new utterances are part of the topic at hand (under the same conversation root) and are sorted by recency
        if (len(correct_list_order) != len(conversation_ids_list)):
            difference_in_sets = set(conversation_ids_list).difference(correct_list_order)
            timestamp_sorted_difference = sort_by_timestamp(list(difference_in_sets), conversation_corpus)
            correct_list_order.extend(timestamp_sorted_difference)
    return correct_list_order


And so, we can compute the correct order of utterances in each randomly selected conversation.

In [50]:
for conversation in random_conversations:
    conversation_ids_list = conversation.get_utterance_ids()
    print ('Original Order of IDs:' + str(conversation_ids_list))
    print('Correct Order of IDs:' + str(find_correct_order(conversation_ids_list, wikiconv_corpus)))
    print ('\n')

Original Order of IDs:['1689329.121.121', '1689329.134.121', '1794219.672.672', '1861940.831.831', '2655439.1016.1016']
Correct Order of IDs:['1689329.121.121', '1861940.831.831', '2655439.1016.1016', '1689329.134.121', '1794219.672.672']


Original Order of IDs:['3660768.329714.519531', '3660768.329714.519574']
Correct Order of IDs:['3660768.329714.519531', '3660768.329714.519574']


Original Order of IDs:['2331863.9948.9977', '2330378.10002.9977']
Correct Order of IDs:['2331863.9948.9977', '2330378.10002.9977']




Print out the final form of the conversations

In [51]:
#Print the conversation text from the list of conversation ids
def print_final_conversation(random_conversations, conversation_corpus):
    for conversation in random_conversations:
        print(wikipedia_link_info(conversation))
        conversation_ids_list = conversation.get_utterance_ids()
        #First correctly reorder the comments
        ordered_list = find_correct_order(conversation_ids_list, conversation_corpus)
        #For each utterance, print the text present if the utterance has not been deleted
        for utterance_id in ordered_list:
            utterance_value = conversation_corpus.get_utterance(utterance_id)
            if (utterance_value.text != " "):
                print (utterance_value.text)
                date_time_val = datetime.fromtimestamp(utterance_value.timestamp).strftime('%H:%M %d-%m-%Y')
                formatted_user_name = "--" + str(utterance_value.speaker.id) + "  " + str(date_time_val)
                print (formatted_user_name)
        print ('\n\n')

In [52]:
print_final_conversation(random_conversations, wikiconv_corpus)

https://en.wikipedia.org/w/index.php?title=user_talk:Frank_Jacobsen
Datentyp
--128.176.114.194  18:19 29-06-2004
Hallo Frank! Ich habe gerade eine Anmerkung auf der Diskussionsseite zu C# eingetragen. Vielleicht schaust du mal vorbei.  Zäh-Scharp 21:29, 14. Jul 2004 (CEST)
--Zäh-Scharp  19:29 14-07-2004
Hallo Frank! Ich habe hier weiteres zum Thema geschrieben. Zäh-Scharp 18:59, 15. Aug 2004 (CEST)
--Zäh-Scharp  16:59 15-08-2004
Ein Datentyp und ein abstrakter Datentyp sind nicht dasselbe. Statt eines falschen Redirects (vgl Wikipedia:Redirect sollte daher vielmehr der Abschnitt sogar ausgelagert und sinnvoll verlinkt werden. Lange Texte, die gleich mehrere Sachen definieren, verhindern Interwikilinks und sorgen regelmäßig für das Mehrfachanlegen von Artikeln. Kleine Artikel sorgen für das schnelle Auffinden von Informationen. Daher lieber kleine Artikel mit abgeschlossenem Inhalt. 128.176.114.194 20:19, 29. Jun 2004 (CEST)
--128.176.114.194  18:19 29-06-2004
Wir sollten diese Diskussi

Let's create a compact method to change the default values easily

In [53]:
def change_defaults_print_final(conversation_list, number_of_conversations, conversation_min_length,  
                                conversation_corpus):
    random_conversations = print_random_conversations(conversation_list, number_of_conversations_to_print,
                                                     conversation_min_length, wikiconv_corpus)
    print_final_conversation(random_conversations, conversation_corpus)

In [54]:
conversation_list = list(wikiconv_corpus.iter_conversations())
number_of_conversations_to_print = 1
conversation_min_length = 2
#Refresher on where the wikiconv_corpus  is defined
# corpus_path = "/Users/adityajha/Desktop/ConvoKit-master/second_set/conversation_corpus_year_2015"
# wikiconv_corpus = Corpus(filename=corpus_path)

change_defaults_print_final(conversation_list, number_of_conversations_to_print, conversation_min_length,
                            wikiconv_corpus)

https://en.wikipedia.org/w/index.php?title=talk:Christer_Pettersson
7-stellig
--Stern  09:11 30-09-2004
Solna liegt bei Stockholm, ungefähr wie Potsdam bei Berlin liegt. Aus entfernter Perspektive mag man Probleme haben, diese beiden schwedischen Städte zu unterscheiden. Dort, in Solna, liegt jedenfalls das hier aktuelle Krankenhaus.Elchjagd 09:22, 11. Okt 2004 CEST)
Meiner Tageszeitung entnehme ich heute, dass die Obduktion ergeben hat, dass Christer Pettersson sich die Schädelverletzung bei einem Fall auf Pflastersteine infolge eines epileptischen Anfalles zuzog. Bei dem schwer alkoholisierten Patienten traten dann weitere Komplikationen ein, die zum Tode führten.Elchjagd 08:26, 13. Okt 2004 (CEST)
--Elchjägerin  06:26 13-10-2004
Nach einer Meldung der "welt" vom 30. Sept. 2004 starb er in Stockholm? Eugen Ettelt
--Eugen Ettelt  19:23 01-10-2004
Bisher ist in der schwedischen Presse nur von einer schweren Schädelverletzung die Rede gewesen, infolge derer Pettersson ins Krankenhaus ka

Finally, we will create a method to print out the final comment and the intermediate steps in the conversation

In [89]:
def sort_changes_by_timestamp(modification_list, deletion_list, restoration_list,  original_utterance):
    text_time_tuple_list = []
    if (original_utterance is not None):
        text_time_original  = (original_utterance['text'],original_utterance['timestamp'],
                           original_utterance['speaker']['id'], 'original')
        text_time_tuple_list.append(text_time_original)
        

    for utterance in modification_list:
        text_time= (utterance['text'], utterance['timestamp'],
                    utterance['speaker']['id'], 'modification')
        text_time_tuple_list.append(text_time)
    
    for utterance in deletion_list:
        text_time= ('', utterance['timestamp'],
                    utterance['speaker']['id'], 'deletion')
        text_time_tuple_list.append(text_time)
        
    for utterance in restoration_list:
        text_time= (utterance['text'], utterance['timestamp'],
                    utterance['speaker']['id'], 'restoration')
        text_time_tuple_list.append(text_time)
            
    text_time_tuple_list.sort(key=lambda x: x[1])
    #text_time_tuple_list.reverse()
    
    
    
    return text_time_tuple_list
        
    

In [90]:
def print_intermediate_conversation(random_conversations, conversation_corpus):
    for conversation in random_conversations:
        conversation_ids_list = conversation.get_utterance_ids()
        #First correctly reorder the comments
        ordered_list = find_correct_order(conversation_ids_list, conversation_corpus)
        #For each utterance, print the text present if the utterance has not been deleted
        for utterance_id in ordered_list:
            utterance_value = conversation_corpus.get_utterance(utterance_id)
            if (utterance_value.text != " "):
                final_comment =  utterance_value.text
                date_time_val = datetime.fromtimestamp(utterance_value.timestamp).strftime('%H:%M %d-%m-%Y')
                formatted_user_name = "--" + str(utterance_value.speaker.id) + "  " + str(date_time_val)
                
        
                final_timestamp = utterance_value.timestamp
                modification_list = utterance_value.meta['modification']
                deletion_list = utterance_value.meta['deletion']
                restoration_list = utterance_value.meta['restoration']
                
                sorted_timestamps = sort_changes_by_timestamp(modification_list, deletion_list, restoration_list,
                                                             utterance_value.meta['original'])
                
                if (len(sorted_timestamps)>0):
                    print(wikipedia_link_info(conversation))
                    print ('Final Comment')
                    print (final_comment)
                    print (formatted_user_name)
                    
                    for value in sorted_timestamps:
                        print ('\n')
                        print (value[3])
                        print (value[0])
                        formatted_user_name = "--" + str(value[2]) + "  " + str(datetime.fromtimestamp(float(value[1])).strftime('%H:%M %d-%m-%Y'))
                        #str(datetime.fromtimestamp(value[1]).strftime('%H:%M %d-%m-%Y'))
                        print (formatted_user_name)

                        

Our method to quikcly print out intermediate conversations defined below (only conversations with modification, deletion and restoration conversations  are shown)

In [91]:
def change_defaults_print_intermediate(conversation_list, number_of_conversations, conversation_min_length,  
                                conversation_corpus):
    random_conversations = print_random_conversations(conversation_list, number_of_conversations_to_print,
                                                     conversation_min_length, wikiconv_corpus)
    print_intermediate_conversation(random_conversations, conversation_corpus)

Here, the flow of different conversations  is shown with the final comment first displayed and the corresponding actions that have occurred from earliest to latest actions

In [92]:
conversation_list = list(wikiconv_corpus.iter_conversations())
number_of_conversations_to_print = 10
conversation_min_length = 3

change_defaults_print_intermediate(conversation_list, number_of_conversations_to_print, conversation_min_length,
                            wikiconv_corpus)

https://en.wikipedia.org/w/index.php?title=wikipedia_talk:Hauptseite/alt
Final Comment
DOne:)^°^   12:25, 6. Nov 2004 (CET)
--82.82.68.38  16:05 27-11-2004


original
DOne:)^°^   12:25, 6. Nov 2004 (CET)
--Nerd  11:25 06-11-2004
https://en.wikipedia.org/w/index.php?title=wikipedia_talk:Hauptseite/alt
Final Comment
Thanks!!!Gregor Helms 15:30, 6. Nov 2004 (CET)
--82.82.68.38  16:05 27-11-2004


original
Thanks!!!Gregor Helms 15:30, 6. Nov 2004 (CET)
--GregorHelms  14:30 06-11-2004
https://en.wikipedia.org/w/index.php?title=wikipedia_talk:Hauptseite/alt
Final Comment
JOP, dafür, rein damit (kein Regel ohne Ausnhame:)^°^   09:18, 14. Nov 2004 (CET)
--82.82.68.38  16:05 27-11-2004


original
JOP, dafür, rein damit (kein Regel ohne Ausnhame:)^°^   09:18, 14. Nov 2004 (CET)
--Nerd  08:18 14-11-2004
